# Generating type 4 clones with LLMs

## Generation

In [60]:
import os, re, json, textwrap, tempfile, subprocess, sys, uuid, random
import requests

DATASET_PATH = "../dataset/bigcodebench_normalized.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json"
OLLAMA_MODEL = "llama3.1:latest"   

# Generation settings
LLM_OPTS = {
    "temperature": 0.6,
    "top_p": 0.95,
    "repeat_penalty": 1.05,
    "num_predict": 768,   
}

N_ENTRIES = 4
CLONES_PER_ENTRY = 2  

In [61]:
def call_ollama_chat(messages, model=OLLAMA_MODEL, options=LLM_OPTS):
    """
    Call Ollama's /api/chat with role-based messages.
    Returns raw string content from the assistant.
    """
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": messages,
            "stream": False,
            "options": options
        },
        timeout=600
    )
    resp.raise_for_status()
    data = resp.json()
    return data["message"]["content"]

def extract_python_code(text: str) -> str:
    """
    Extract the first ```python ... ``` fenced block;
    if none found, return the whole text.
    """
    m = re.search(r"```python\s*(.*?)```", text, flags=re.S)
    if m:
        return m.group(1).strip()
    m = re.search(r"```\s*(.*?)```", text, flags=re.S) 
    return (m.group(1).strip() if m else text.strip())

def force_function_name(code: str, expected="task_func"):
    """
    Ensure the function is named `expected`.
    If the model wrote a different name, rename the top-level function.
    """
    import ast, astor
    try:
        tree = ast.parse(textwrap.dedent(code))
        for node in tree.body:
            if isinstance(node, ast.FunctionDef):
                node.name = expected
                break
        ast.fix_missing_locations(tree)
        return astor.to_source(tree)
    except Exception:
        return code  # if parsing fails, return as-is; validation will catch issues


In [62]:
# import tempfile
# import subprocess
# import os
# import sys
# import textwrap
# import json

# def validate_with_unittest(code: str, tests: list, detailed: bool = False):
#     """
#     Run the provided Python code + unittest tests in a temporary script.

#     Args:
#         code (str): The Python code to test.
#         tests (list): List of test strings (unittest test cases).
#         detailed (bool): If True, returns a dict {test_name: True/False}.
#                          If False, returns True/False for overall success.

#     Returns:
#         bool or dict: Overall success (False if any test fails) or dict of per-test results.
#     """
#     # Dedent code and tests
#     code_d = textwrap.dedent(code)
#     tests_d = "\n\n".join(textwrap.dedent(t) for t in tests)

#     # Patch unittest to collect results
#     patch_code = r"""
# import unittest, json

# class JSONTestResult(unittest.TextTestResult):
#     def __init__(self, *a, **kw):
#         super().__init__(*a, **kw)
#         self.results = {}

#     def addSuccess(self, test):
#         super().addSuccess(test)
#         self.results[str(test)] = True

#     def addFailure(self, test, err):
#         super().addFailure(test, err)
#         self.results[str(test)] = False

#     def addError(self, test, err):
#         super().addError(test, err)
#         self.results[str(test)] = False

# class JSONTestRunner(unittest.TextTestRunner):
#     def __init__(self, *a, **kw):
#         super().__init__(*a, resultclass=JSONTestResult, **kw)

#     def run(self, test):
#         result = super().run(test)
#         # Print JSON so we can parse it in Python
#         print("UNITTEST_RESULTS_JSON:" + json.dumps(result.results))
#         return result

# unittest.TextTestRunner = JSONTestRunner
# """

#     # Combine everything into one script
#     full_script = f"""
# {code_d}

# {tests_d}

# {patch_code}

# if __name__ == "__main__":
#     unittest.main()
# """

#     try:
#         # Write to a temporary .py file
#         with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
#             f.write(full_script)
#             tmp_path = f.name

#         # Execute the temp script
#         res = subprocess.run(
#             [sys.executable, tmp_path],
#             stdout=subprocess.PIPE,
#             stderr=subprocess.PIPE,
#             text=True
#         )

#         os.remove(tmp_path)

#         # Look for the JSON output from our patched runner
#         results = {}
#         for line in res.stdout.splitlines():
#             if line.startswith("UNITTEST_RESULTS_JSON:"):
#                 results = json.loads(line.split(":", 1)[1])
#                 break

#         if detailed:
#             return results  # Return per-test dictionary
#         else:
#             return res.returncode == 0  # Overall True/False

#     except Exception as e:
#         print("Validation error:", e)
#         return {} if detailed else False


In [63]:
STRATEGY_HINTS = [
    # Keep behavior identical; do shallow refactors
    "- rename locals & args; reorder independent statements; introduce small helper vars; keep library calls and side-effects identical.",
    "- replace simple loops with list/dict comprehensions where safe; adjust arithmetic with equivalent identities; keep signature & imports.",
    "- wrap small expressions into temporary variables; change exception handling style without changing raised exceptions.",
]

SYSTEM_PROMPT = """You are a careful Python refactoring engine.
You produce a semantically equivalent variant (Type-4 clone) of the given function.
Rules:
- Output ONLY Python code in a single fenced block.
- Define exactly one function named `task_func` with the correct signature for the tests.
- Keep the same external behavior, side-effects, and library usage (imports allowed).
- Do NOT hardcode any test data or specific URLs or values from tests.
- Keep I/O contract identical (same return types, shapes, and exceptions).
"""

def build_user_prompt(original_body: str, description: str, libs: list, tests_snippet: str, strategy_hint: str) -> str:
    return f"""
You will be shown:
1) A short description and allowed libraries.
2) The original function BODY (not including the def line).
3) An excerpt of the unit tests (for signature and behavior cues). Do not overfit.

Description:
{description}

Allowed/expected libraries (may import as needed): {libs}

Original function BODY (indentation represents inside the function):
{textwrap.dedent(original_body).strip()}

Unit test excerpt (do not hardcode values; just infer signature/contract):
{textwrap.shorten(textwrap.dedent(tests_snippet), width=2000, placeholder=" ... ")}


Your task:
- Emit a semantically equivalent implementation named `task_func`.
- Keep side effects and external calls intact where visible (e.g., urllib/os/json/pandas usage).
- {strategy_hint}

Return ONLY the code in a single ```python fenced block.
"""


In [64]:
import os
import json  

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

sample = data[:N_ENTRIES]

results = []
for i, entry in enumerate(sample, 1):
    print(f"\n===Generating clones for Entry {i}/{len(sample)} | id={entry['id']} ===")
    clones = []

    original_body = entry["original_code"]
    tests_list    = entry["test"]
    description   = entry.get("description", "")
    libs          = entry.get("metadata", {}).get("libs", [])

    tests_snippet = tests_list[0] if tests_list else ""

    for k in range(CLONES_PER_ENTRY):
        hint = STRATEGY_HINTS[k % len(STRATEGY_HINTS)]
        user_prompt = build_user_prompt(original_body, description, libs, tests_snippet, hint)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ]

        try:
            raw = call_ollama_chat(messages)
            code = extract_python_code(raw)
            code = force_function_name(code, expected="task_func")

            clones.append({
                "transformation": f"LLM/{OLLAMA_MODEL}",
                "strategy_hint": hint,
                "code": code
            })
        except Exception as e:
            print(f"  Error generating clone {k+1}: {e}")

    results.append({
        "id": entry["id"],
        "language": entry["language"],
        "description": description,
        "metadata": entry.get("metadata", {}),
        "original_code": original_body,
        "test": tests_list,
        "clones": clones
    })

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)



===Generating clones for Entry 1/4 | id=b7dd7eea-403d-4e91-af7a-f4ccce80f14e ===

===Generating clones for Entry 2/4 | id=241257cf-7dd4-4621-b4f7-c68e81426484 ===

===Generating clones for Entry 3/4 | id=a8f7aee6-c252-412b-9614-4b18e43682eb ===

===Generating clones for Entry 4/4 | id=2ead070e-1468-4956-983d-9c9f7a79831c ===


## Running the tests on the clones

In [65]:
import unittest
import tempfile
import textwrap
import importlib.util
import sys
import os

class TrackingTestResult(unittest.TextTestResult):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.successes = []

    def addSuccess(self, test):
        super().addSuccess(test)
        self.successes.append(test)

def validate_with_unittest(code: str, tests: list) -> dict:
    """
    Run code + tests and return per-test results as {test_name: "PASS"/"FAIL"}.
    Minimal changes from your original function.
    """
    code_d = textwrap.dedent(code)
    tests_d = "\n\n".join(textwrap.dedent(t) for t in tests)

    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code_d + "\n\n" + tests_d)
        tmp_path = f.name

    try:
        # Load module dynamically
        spec = importlib.util.spec_from_file_location("tmp_module", tmp_path)
        tmp_module = importlib.util.module_from_spec(spec)
        sys.modules["tmp_module"] = tmp_module
        spec.loader.exec_module(tmp_module)

        loader = unittest.TestLoader()
        suite = loader.loadTestsFromModule(tmp_module)

        # Run with our tracking TestResult
        stream = open(os.devnull, 'w')  # suppress default TextTestRunner output
        runner = unittest.TextTestRunner(stream=stream, resultclass=TrackingTestResult)
        result = runner.run(suite)

        test_results = {}
        for test_case, _ in result.failures + result.errors:
            test_results[str(test_case)] = "FAIL"
        for test_case in result.successes:
            test_results[str(test_case)] = "PASS"

        return test_results

    except Exception as e:
        print("Validation error:", e)
        return {}
    finally:
        os.remove(tmp_path)


In [66]:
import os
import json 

with open(OUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

results = []
for i, entry in enumerate(data, 1):
    print(f"\n=== Testing Entry {i}/{len(data)} | id={entry['id']} ===")
    clones = []

    tests_list = entry["test"]

    for k, clone in enumerate(entry.get("clones", []),1):
        try:
            code = clone["code"]
            # Get individual test results
            test_results = validate_with_unittest(code, tests_list)
            clone["test_results"] = test_results

            # Print summary
            passed = sum(1 for v in test_results.values() if v=="PASS")
            total = len(test_results)
            print(f"  Clone {k}: {passed}/{total} tests passed")

            clones.append(clone)
        except Exception as e:
            print(f"  Error testing clone {k}: {e}")
            clone["test_results"] = {}
            clones.append(clone)

    entry["clones"] = clones
    results.append(entry)

# Save dataset with test results
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Done. Saved dataset with test results to {OUT_PATH}")



=== Testing Entry 1/4 | id=b7dd7eea-403d-4e91-af7a-f4ccce80f14e ===
  Clone 1: 10/10 tests passed
  Clone 2: 10/10 tests passed

=== Testing Entry 2/4 | id=241257cf-7dd4-4621-b4f7-c68e81426484 ===
  Clone 1: 3/3 tests passed
  Clone 2: 3/3 tests passed

=== Testing Entry 3/4 | id=a8f7aee6-c252-412b-9614-4b18e43682eb ===
  Clone 1: 5/5 tests passed
  Clone 2: 5/5 tests passed

=== Testing Entry 4/4 | id=2ead070e-1468-4956-983d-9c9f7a79831c ===
  Clone 1: 5/5 tests passed
  Clone 2: 5/5 tests passed

✅ Done. Saved dataset with test results to ../results/bigcodebench_llm_clones.json


## Checking clone type

In [67]:
import json
from codebleu import calc_codebleu  

# === Load clones dataset ===
with open(OUT_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# === Compute CodeBLEU for each clone ===
for idx, entry in enumerate(dataset):
    original_code = entry["original_code"]
    clones = entry.get("clones", [])
    for clone in clones:
        try:
            clone_code = clone["code"]
            # CodeBLEU expects lists: (refs, hyps, lang)
            score_dict = calc_codebleu([original_code], [clone_code], lang="python", weights=(0.25, 0.25, 0.25, 0.25), tokenizer=None)
            codebleu_score = score_dict["codebleu"]
            
            # Store in results
            clone["metrics"] = {"codebleu": codebleu_score}
            print(f"[Entry {idx}] Clone by {clone['transformation']} → CodeBLEU: {codebleu_score:.4f}")
        
        except Exception as e:
            print(f"Error scoring clone for entry {idx}: {e}")
            clone["metrics"] = {"codebleu": None}

# === Save dataset with metrics ===
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print("\n✅ Finished scoring all clones with CodeBLEU.")


[Entry 0] Clone by LLM/llama3.1:latest → CodeBLEU: 0.5231
[Entry 0] Clone by LLM/llama3.1:latest → CodeBLEU: 0.4327
[Entry 1] Clone by LLM/llama3.1:latest → CodeBLEU: 0.6705
[Entry 1] Clone by LLM/llama3.1:latest → CodeBLEU: 0.6887
[Entry 2] Clone by LLM/llama3.1:latest → CodeBLEU: 0.7000
[Entry 2] Clone by LLM/llama3.1:latest → CodeBLEU: 0.9462
[Entry 3] Clone by LLM/llama3.1:latest → CodeBLEU: 0.6628
[Entry 3] Clone by LLM/llama3.1:latest → CodeBLEU: 0.9532

✅ Finished scoring all clones with CodeBLEU.
